# Hospital Readmissions — XAI/MLI Pipeline
**LangGraph agentic workflow adapted from the Fraud Detection pipeline.**

### Pipeline Architecture
```
START
  → eda_agent                   (8 Plotly EDA charts)
  → clean_feature_engineering   (impute · encode · SMOTE · scale)
  → train_models_agent          (LR · RF · LightGBM · XGBoost)
  → evaluate_models_agent       (accuracy · precision · recall · F1 · AUC)
  → select_model_agent          (weighted composite score)
  → shap_agent                  (global summary + local waterfall)
  → lime_agent                  (local explanations for 3 patients)
  → pdp_agent                   (partial dependence plots)
  → decision_agent              (clinical risk triage per patient)
  → business_agent              (ROI, cost savings, downstream recommendations)
END
```

**Dataset:** `dubradave/hospital-readmissions` (Kaggle)  
**Target:** `readmitted` — 1 if patient readmitted within 30 days, 0 otherwise  
**Business question:** *Which patients are at highest risk of readmission, and why?*

## Cell 0 — Colab Setup (clone repo + Kaggle credentials)

In [ ]:
# ── Colab Setup: clone repo + Kaggle credentials ──────────────────────────────
import os

# Clone repo if not already inside it
if not os.path.exists('XAI_hospital-readmissions'):
    !git clone https://github.com/00vera/XAI_hospital-readmissions.git
if os.path.basename(os.getcwd()) != 'XAI_hospital-readmissions':
    os.chdir('XAI_hospital-readmissions')

# Upload kaggle.json
# Get it from: kaggle.com → your profile → Settings → API → Create New Token
from google.colab import files
uploaded = files.upload()   # select kaggle.json when prompted

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

print('Colab setup complete!')


## Cell 1 — Install Dependencies

In [ ]:
!pip install langgraph langchain-core pandas scikit-learn numpy imbalanced-learn \n             lightgbm xgboost shap lime plotly matplotlib kagglehub -q


## Cell 2 — Imports & Data Load
> **Note:** Run your own prep cell first (kagglehub download) then pass `raw_df` below.

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.io as pio

from typing import TypedDict, Optional, Dict, Any, List

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import partial_dependence
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

# imbalanced-learn
from imblearn.over_sampling import SMOTE

# boosting
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# XAI
import shap
import lime
import lime.lime_tabular

# LangGraph
from langgraph.graph import StateGraph, START, END

print('✓ All imports successful')

## Cell 3 — Load Dataset
*(Assumes `raw_df` already loaded from your prep cell. If not, run kagglehub download here.)*

In [ ]:
import kagglehub

path = kagglehub.dataset_download('dubradave/hospital-readmissions')
filename = [f for f in os.listdir(path) if f.endswith('.csv')][0]
raw_df = pd.read_csv(os.path.join(path, filename))

df = raw_df.copy()

# Binarise the target
# Original values: '<30', '>30', 'NO'
# Business goal: predict readmission within 30 days
df['readmitted'] = (df['readmitted'] == '<30').astype(int)

print(f'Dataset shape : {df.shape}')
print(f'Target distribution:
{df["readmitted"].value_counts()}
')
print(f'Readmission rate: {df["readmitted"].mean()*100:.1f}%')
df.head(3)


## Cell 4 — AgentState Definition

In [ ]:
class AgentState(TypedDict):
    """Shared state passed through every node of the hospital readmissions pipeline."""
    # ── upstream data ──
    raw_data:               pd.DataFrame
    target_column:          str
    eda_charts_path:        Optional[str]
    # ── feature engineering ──
    X_train:                Optional[pd.DataFrame]
    X_test:                 Optional[pd.DataFrame]
    y_train:                Optional[pd.Series]
    y_test:                 Optional[pd.Series]
    feature_columns:        Optional[list]
    # ── training ──
    trained_models:         Optional[Dict[str, Any]]
    training_summary:       Optional[Dict[str, Any]]
    candidate_model_names:  Optional[list]
    # ── evaluation ──
    evaluation_results:     Optional[Dict[str, Any]]
    best_model_name:        Optional[str]
    # ── selection ──
    selected_model_name:        Optional[str]
    selected_model:             Optional[Any]
    selection_rationale:        Optional[str]
    selection_scores:           Optional[Dict[str, float]]
    selection_comparison_path:  Optional[str]
    # ── XAI ──
    shap_summary_path:      Optional[str]
    shap_values:            Optional[Any]
    lime_explanations:      Optional[List[Dict]]
    pdp_paths:              Optional[List[str]]
    # ── decision + business ──
    patient_risk_decisions: Optional[List[Dict]]
    business_report:        Optional[str]

print('✓ AgentState defined')

## Cell 5 — EDA Agent

In [ ]:
def eda_agent(state: AgentState) -> AgentState:
    """
    Performs Exploratory Data Analysis on the hospital readmissions dataset.

    Charts generated:
    1. readmission_distribution.html    — Pie chart of 30-day readmission rate
    2. time_in_hospital_hist.html       — Histogram: days in hospital by outcome
    3. num_medications_hist.html        — Histogram: medication count by outcome
    4. num_diagnoses_box.html           — Box plot: diagnoses by outcome
    5. age_readmission_bar.html         — Bar: readmission rate by age group
    6. num_lab_procedures_violin.html   — Violin: lab procedures by outcome
    7. admission_type_bar.html          — Bar: readmission by admission type
    8. correlation_heatmap.html         — Heatmap of numeric feature correlations
    """
    print('[EDA] Generating exploratory data analysis charts...')
    pio.templates.default = 'plotly_dark'

    df = state['raw_data'].copy()
    target = state['target_column']

    eda_path = 'eda_outputs'
    os.makedirs(eda_path, exist_ok=True)

    # ── CHART 1: Readmission distribution ─────────────────────────────────────
    counts = df[target].value_counts().reset_index()
    counts.columns = ['Readmitted', 'Count']
    counts['Readmitted'] = counts['Readmitted'].map({1: 'Readmitted (<30d)', 0: 'Not Readmitted'})
    fig1 = px.pie(counts, values='Count', names='Readmitted',
                  title='30-Day Hospital Readmission Distribution',
                  color_discrete_sequence=['#E74C3C', '#2ECC71'])
    fig1.write_html(f'{eda_path}/readmission_distribution.html')
    fig1.show()
    print('  ✓ readmission_distribution.html')

    # ── CHART 2: Time in hospital histogram ────────────────────────────────────
    df_plot = df.copy()
    df_plot['Outcome'] = df_plot[target].map({1: 'Readmitted', 0: 'Not Readmitted'})
    fig2 = px.histogram(df_plot, x='time_in_hospital', color='Outcome',
                         title='Length of Stay by Readmission Outcome',
                         nbins=14, barmode='overlay',
                         labels={'time_in_hospital': 'Days in Hospital'})
    fig2.write_html(f'{eda_path}/time_in_hospital_hist.html')
    fig2.show()
    print('  ✓ time_in_hospital_hist.html')

    # ── CHART 3: Num medications histogram ────────────────────────────────────
    fig3 = px.histogram(df_plot, x='num_medications', color='Outcome',
                         title='Number of Medications by Readmission Outcome',
                         nbins=30, barmode='overlay',
                         labels={'num_medications': 'Number of Medications'})
    fig3.write_html(f'{eda_path}/num_medications_hist.html')
    fig3.show()
    print('  ✓ num_medications_hist.html')

    # ── CHART 4: Number of diagnoses box plot ──────────────────────────────────
    fig4 = px.box(df_plot, x='Outcome', y='number_diagnoses',
                   title='Number of Diagnoses by Readmission Outcome',
                   labels={'number_diagnoses': 'Number of Diagnoses'})
    fig4.write_html(f'{eda_path}/num_diagnoses_box.html')
    fig4.show()
    print('  ✓ num_diagnoses_box.html')

    # ── CHART 5: Readmission rate by age group ────────────────────────────────
    if 'age' in df.columns:
        age_readmit = df.groupby('age')[target].mean().reset_index()
        age_readmit.columns = ['Age Group', 'Readmission Rate']
        age_readmit['Readmission Rate'] = age_readmit['Readmission Rate'] * 100
        fig5 = px.bar(age_readmit, x='Age Group', y='Readmission Rate',
                       title='30-Day Readmission Rate by Age Group (%)',
                       labels={'Readmission Rate': 'Readmission Rate (%)'},
                       color='Readmission Rate', color_continuous_scale='Reds')
        fig5.write_html(f'{eda_path}/age_readmission_bar.html')
        fig5.show()
        print('  ✓ age_readmission_bar.html')

    # ── CHART 6: Lab procedures violin ────────────────────────────────────────
    fig6 = px.violin(df_plot, x='Outcome', y='num_lab_procedures', box=True,
                      title='Lab Procedures by Readmission Outcome',
                      labels={'num_lab_procedures': 'Number of Lab Procedures'})
    fig6.write_html(f'{eda_path}/num_lab_procedures_violin.html')
    fig6.show()
    print('  ✓ num_lab_procedures_violin.html')

    # ── CHART 7: Admission type bar ───────────────────────────────────────────
    if 'admission_type_id' in df.columns:
        adm_readmit = df.groupby('admission_type_id')[target].mean().reset_index()
        adm_readmit.columns = ['Admission Type ID', 'Readmission Rate']
        adm_readmit['Readmission Rate'] = adm_readmit['Readmission Rate'] * 100
        fig7 = px.bar(adm_readmit, x='Admission Type ID', y='Readmission Rate',
                       title='Readmission Rate by Admission Type (%)',
                       color='Readmission Rate', color_continuous_scale='Blues')
        fig7.write_html(f'{eda_path}/admission_type_bar.html')
        fig7.show()
        print('  ✓ admission_type_bar.html')

    # ── CHART 8: Correlation heatmap ──────────────────────────────────────────
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    key_cols = [c for c in [
        'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_outpatient', 'number_emergency',
        'number_inpatient', 'number_diagnoses', target
    ] if c in numeric_cols]
    corr = df[key_cols].corr()
    fig8 = px.imshow(corr, title='Feature Correlation Heatmap',
                      color_continuous_scale='RdBu_r',
                      labels=dict(color='Correlation'))
    fig8.write_html(f'{eda_path}/correlation_heatmap.html')
    fig8.show()
    print('  ✓ correlation_heatmap.html')

    print(f'\n[EDA] ✓ Complete — 8 charts saved to {eda_path}/')

    return {
        **state,
        'eda_charts_path': eda_path,
    }

print('✓ eda_agent defined')

## Cell 6 — Feature Engineering Agent

In [ ]:
def clean_feature_engineering(state: AgentState) -> AgentState:
    """
    Hospital-specific feature engineering pipeline:
    1. Drop ID / leakage columns
    2. Impute missing values (median for numeric, 'missing' for categorical)
    3. Remove duplicates and invalid rows
    4. Engineer domain features (medication burden, readmission history score)
    5. One-hot encode categoricals
    6. Stratified train/test split (80/20)
    7. StandardScale numeric features (fit on train only)
    8. Apply SMOTE on training set only
    """
    print('[FEAT] Starting feature engineering...')

    df = state['raw_data'].copy()
    target_col = state['target_column']

    # ── 1. Drop ID / leakage / high-cardinality columns ──────────────────────
    drop_cols = [
        'encounter_id', 'patient_nbr',       # identifiers
        'examide', 'citoglipton',             # single-value columns
        'weight',                             # >95% missing
        'payer_code', 'medical_specialty',    # high missingness / leakage
    ]
    drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=drop_cols)
    print(f'[CLEAN] Dropped {len(drop_cols)} irrelevant columns: {drop_cols}')

    # ── 2. Replace '?' with NaN ───────────────────────────────────────────────
    df = df.replace('?', np.nan)

    # ── 3. Impute missing values ──────────────────────────────────────────────
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_feat = [c for c in numeric_cols if c != target_col]
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

    if numeric_feat:
        num_imp = SimpleImputer(strategy='median')
        df[numeric_feat] = num_imp.fit_transform(df[numeric_feat])
        print(f'[IMPUTE] Imputed {len(numeric_feat)} numeric columns with median')

    if categorical_cols:
        cat_imp = SimpleImputer(strategy='constant', fill_value='missing')
        df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])
        print(f'[IMPUTE] Imputed {len(categorical_cols)} categorical columns')

    # ── 4. Remove duplicates ──────────────────────────────────────────────────
    before = len(df)
    df = df.drop_duplicates()
    print(f'[CLEAN] Removed {before - len(df)} duplicate rows')

    # ── 5. Engineer domain features ───────────────────────────────────────────
    # Medication burden: normalised medication load
    if 'num_medications' in df.columns and 'time_in_hospital' in df.columns:
        df['meds_per_day'] = df['num_medications'] / (df['time_in_hospital'] + 1e-9)
        print('[FEATURE] Created meds_per_day')

    # Prior healthcare utilisation score
    util_cols = ['number_outpatient', 'number_emergency', 'number_inpatient']
    present = [c for c in util_cols if c in df.columns]
    if present:
        df['prior_utilisation'] = df[present].sum(axis=1)
        print('[FEATURE] Created prior_utilisation')

    # Log-transform skewed numerics
    for col in ['num_medications', 'num_lab_procedures', 'number_diagnoses']:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(df[col])
    print('[FEATURE] Log-transformed skewed numeric features')

    # ── 6. Separate features & target ─────────────────────────────────────────
    if target_col not in df.columns:
        raise ValueError(f'Target column "{target_col}" not found')

    y = df[target_col].copy()
    X = df.drop(columns=[target_col])

    # ── 7. One-hot encode ──────────────────────────────────────────────────────
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    if cat_cols:
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
        print(f'[ENCODE] One-hot encoded {len(cat_cols)} categorical columns → {X.shape[1]} features')

    # ── 8. Train/test split ───────────────────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    print(f'[SPLIT] Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')

    # ── 9. Scale ──────────────────────────────────────────────────────────────
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    print(f'[SCALE] Scaled {len(num_cols)} numeric features')

    # ── 10. SMOTE on training set only ────────────────────────────────────────
    before_dist = y_train.value_counts().sort_index().to_dict()
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    after_dist = y_train.value_counts().sort_index().to_dict()
    print(f'[SMOTE] Before: {before_dist} → After: {after_dist}')

    feature_columns = X_train.columns.tolist()
    print(f'\n[FEAT] ✓ Complete — {len(feature_columns)} features | '
          f'X_train: {X_train.shape} | X_test: {X_test.shape}')

    return {
        **state,
        'X_train':        X_train,
        'X_test':         X_test,
        'y_train':        y_train,
        'y_test':         y_test,
        'feature_columns': feature_columns,
    }

print('✓ clean_feature_engineering defined')

## Cell 7 — Train Models Agent

In [ ]:
def train_models_agent(state: AgentState) -> AgentState:
    """
    Train 4 candidate models on the SMOTE-balanced training data.
    Models: Logistic Regression, Random Forest, LightGBM, XGBoost.
    """
    print('[TRAIN] Starting model training...\n')

    X_train = state['X_train']
    y_train = state['y_train']

    print(f'[TRAIN] X_train: {X_train.shape}')
    print(f'[TRAIN] y_train distribution: {y_train.value_counts().sort_index().to_dict()}\n')

    candidate_models = {
        'logistic_regression': LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42
        ),
        'random_forest': RandomForestClassifier(
            n_estimators=200, max_depth=12,
            min_samples_split=10, min_samples_leaf=4,
            random_state=42, n_jobs=-1
        ),
        'lightgbm': LGBMClassifier(
            n_estimators=250, max_depth=6,
            learning_rate=0.05, random_state=42,
            verbose=-1
        ),
        'xgboost': XGBClassifier(
            n_estimators=250, max_depth=6,
            learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, objective='binary:logistic',
            eval_metric='logloss', random_state=42,
            n_jobs=-1, verbosity=0
        )
    }

    trained_models = {}
    training_summary = {}

    for name, model in candidate_models.items():
        print(f'[TRAIN] Training {name}...')
        t0 = time.time()
        model.fit(X_train, y_train)
        elapsed = round(time.time() - t0, 2)
        trained_models[name] = model
        training_summary[name] = {
            'model_class': model.__class__.__name__,
            'train_time_seconds': elapsed,
            'n_train_rows': int(X_train.shape[0]),
            'n_features': int(X_train.shape[1]),
            'status': 'trained'
        }
        print(f'        ✓ Done in {elapsed}s')

    print(f'\n[TRAIN] ✓ Trained: {list(trained_models.keys())}')

    return {
        **state,
        'trained_models':        trained_models,
        'training_summary':      training_summary,
        'candidate_model_names': list(trained_models.keys()),
    }

print('✓ train_models_agent defined')

## Cell 8 — Evaluate Models Agent

In [ ]:
def evaluate_models_agent(state: AgentState) -> AgentState:
    """
    Evaluate all trained models on the untouched test set.
    Saves confusion matrices and ROC curves to evaluation_outputs/.
    """
    print('[EVAL] Starting model evaluation...\n')

    trained_models = state['trained_models']
    X_test = state['X_test']
    y_test = state['y_test']

    output_dir = 'evaluation_outputs'
    os.makedirs(output_dir, exist_ok=True)

    evaluation_results = {}

    for model_name, model in trained_models.items():
        print(f'[EVAL] Evaluating {model_name}...')

        y_pred = model.predict(X_test)
        y_score = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

        accuracy  = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)
        auc_roc   = roc_auc_score(y_test, y_score) if y_score is not None else None
        cm        = confusion_matrix(y_test, y_pred)
        report    = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        evaluation_results[model_name] = {
            'accuracy':               float(accuracy),
            'precision':              float(precision),
            'recall':                 float(recall),
            'f1_score':               float(f1),
            'auc_roc':                float(auc_roc) if auc_roc else None,
            'confusion_matrix':       cm.tolist(),
            'classification_report':  report,
        }

        auc_str = f'{auc_roc:.4f}' if auc_roc else 'N/A'
        print(f'        accuracy={accuracy:.4f}, precision={precision:.4f}, '
              f'recall={recall:.4f}, f1={f1:.4f}, auc={auc_str}')

        # Confusion matrix
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(cm, interpolation='nearest')
        ax.figure.colorbar(im, ax=ax)
        ax.set(xticks=np.arange(2), yticks=np.arange(2),
               xticklabels=['Not Readmitted', 'Readmitted'],
               yticklabels=['Not Readmitted', 'Readmitted'],
               ylabel='True Label', xlabel='Predicted Label',
               title=f'Confusion Matrix — {model_name}')
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center')
        plt.tight_layout()
        plt.savefig(f'{output_dir}/{model_name}_confusion_matrix.png', bbox_inches='tight')
        plt.close()

        # ROC curve
        if y_score is not None:
            fpr, tpr, _ = roc_curve(y_test, y_score)
            plt.figure(figsize=(6, 4))
            plt.plot(fpr, tpr, label=f'AUC = {auc_roc:.4f}')
            plt.plot([0, 1], [0, 1], linestyle='--')
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate (Recall)')
            plt.title(f'ROC Curve — {model_name}')
            plt.legend(loc='lower right')
            plt.tight_layout()
            plt.savefig(f'{output_dir}/{model_name}_roc_curve.png', bbox_inches='tight')
            plt.close()

    best_model_name = max(evaluation_results, key=lambda m: evaluation_results[m]['f1_score'])
    print(f'\n[EVAL] ✓ Best model by F1: {best_model_name}')

    return {
        **state,
        'evaluation_results': evaluation_results,
        'best_model_name':    best_model_name,
    }

print('✓ evaluate_models_agent defined')

## Cell 9 — Model Selection Agent
**Weights (healthcare domain):**  
- Recall 40% — a missed readmission = patient harm  
- AUC-ROC 30% — overall discrimination across thresholds  
- F1 20% — balances precision/recall  
- Precision 10% — avoids unnecessary interventions

In [ ]:
def select_model_agent(state: AgentState) -> AgentState:
    """
    Select the best model using a healthcare-weighted composite score.
    Recall weighted highest because missed readmissions = patient harm.
    """
    print('[SELECT] Starting model selection...\n')

    evaluation_results = state['evaluation_results']
    trained_models     = state['trained_models']

    # Healthcare domain weights — recall matters most
    WEIGHTS = {
        'recall':    0.40,   # missed readmission = patient harm
        'auc_roc':   0.30,   # discrimination across all thresholds
        'f1_score':  0.20,   # precision/recall balance
        'precision': 0.10,   # avoid unnecessary interventions
    }

    print('[SELECT] Weighted scoring criteria (hospital domain):')
    for metric, w in WEIGHTS.items():
        print(f'         {metric:<12} {w*100:.0f}%')
    print()

    selection_scores = {}
    for model_name, metrics in evaluation_results.items():
        score = sum(WEIGHTS[m] * (metrics.get(m) or 0.0) for m in WEIGHTS)
        selection_scores[model_name] = round(score, 6)
        print(f'[SELECT] {model_name:<28} composite score = {score:.4f}')

    sorted_models       = sorted(selection_scores, key=lambda m: selection_scores[m], reverse=True)
    selected_model_name = sorted_models[0]
    selected_model      = trained_models[selected_model_name]
    best_metrics        = evaluation_results[selected_model_name]
    runner_up           = sorted_models[1]
    margin              = selection_scores[selected_model_name] - selection_scores[runner_up]

    auc_val = best_metrics.get('auc_roc')
    auc_str = f"{auc_val:.4f}" if auc_val else 'N/A'

    rationale_lines = [
        f"Selected model  : {selected_model_name}",
        f"Composite score : {selection_scores[selected_model_name]:.4f}  "
        f"(runner-up: {runner_up} @ {selection_scores[runner_up]:.4f}, margin: +{margin:.4f})",
        "",
        "Key metric breakdown:",
        f"  Recall    = {best_metrics['recall']:.4f}  (weight 40%) — fraction of actual readmissions caught",
        f"  AUC-ROC   = {auc_str}  (weight 30%) — overall discriminative power",
        f"  F1-score  = {best_metrics['f1_score']:.4f}  (weight 20%) — harmonic mean of precision/recall",
        f"  Precision = {best_metrics['precision']:.4f}  (weight 10%) — avoids unnecessary care interventions",
        "",
        "Business justification:",
        "  In hospital readmission prediction, a missed readmission (false negative)",
        "  directly harms patients and costs hospitals CMS penalty revenue. Recall is",
        "  therefore weighted highest. The selected model achieves the best overall",
        "  balance across all metrics for clinical deployment."
    ]
    selection_rationale = '\n'.join(rationale_lines)

    print(f'\n[SELECT] ✓ Selected: {selected_model_name}')
    print('\n[SELECT] Rationale:')
    print(selection_rationale)

    # ── Comparison chart ──────────────────────────────────────────────────────
    output_dir = 'selection_outputs'
    os.makedirs(output_dir, exist_ok=True)

    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']
    model_names = list(evaluation_results.keys())
    n_models = len(model_names)
    x = np.arange(len(metrics_to_plot))
    width = 0.8 / n_models
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52'][:n_models]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax = axes[0]
    for i, (mn, color) in enumerate(zip(model_names, colors)):
        vals = [evaluation_results[mn].get(m) or 0.0 for m in metrics_to_plot]
        offset = (i - n_models / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=mn, color=color, alpha=0.85,
                      edgecolor='white', linewidth=0.5)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC'], fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_title('Model Comparison — All Metrics', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    ax2 = axes[1]
    sorted_scores = [selection_scores[m] for m in sorted_models]
    bar_colors = ['#2ecc71' if m == selected_model_name else '#95a5a6' for m in sorted_models]
    bars2 = ax2.bar(sorted_models, sorted_scores, color=bar_colors, edgecolor='white',
                    linewidth=0.5, alpha=0.9)
    for bar, score in zip(bars2, sorted_scores):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                 f'{score:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax2.set_ylim(0, max(sorted_scores) * 1.2)
    ax2.set_title('Composite Score Ranking\n(Recall 40% · AUC 30% · F1 20% · Precision 10%)',
                  fontsize=11, fontweight='bold')
    ax2.tick_params(axis='x', labelsize=8)
    ax2.grid(axis='y', linestyle='--', alpha=0.4)
    selected_patch = mpatches.Patch(color='#2ecc71', label=f'Selected: {selected_model_name}')
    ax2.legend(handles=[selected_patch], fontsize=8)

    plt.suptitle('Model Selection Dashboard — Hospital Readmission Pipeline',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    chart_path = f'{output_dir}/model_selection_comparison.png'
    plt.savefig(chart_path, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close()
    print(f'\n[SELECT] Chart saved → {chart_path}')

    with open(f'{output_dir}/selection_rationale.txt', 'w') as f:
        f.write(selection_rationale)
    with open(f'{output_dir}/selection_scores.json', 'w') as f:
        json.dump(selection_scores, f, indent=2)

    print('\n[SELECT] ✓ Model selection complete.\n')

    return {
        **state,
        'selected_model_name':       selected_model_name,
        'selected_model':            selected_model,
        'selection_rationale':       selection_rationale,
        'selection_scores':          selection_scores,
        'selection_comparison_path': chart_path,
    }

print('✓ select_model_agent defined')

## Cell 10 — SHAP Agent (Global + Local XAI)

In [ ]:
def shap_agent(state: AgentState) -> AgentState:
    """
    Generates SHAP explanations for the selected model.

    Outputs:
    - shap_outputs/shap_summary_beeswarm.png  — global feature importance
    - shap_outputs/shap_summary_bar.png       — mean |SHAP| bar chart
    - shap_outputs/shap_waterfall_patient_N.png — local explanation per patient

    Business insight: answers 'WHICH features drive readmission risk GLOBALLY'
    and 'WHY is THIS specific patient at risk?'
    """
    print('[SHAP] Computing SHAP explanations...\n')

    model       = state['selected_model']
    model_name  = state['selected_model_name']
    X_test      = state['X_test']
    feature_columns = state['feature_columns']

    output_dir = 'shap_outputs'
    os.makedirs(output_dir, exist_ok=True)

    # Use a sample for speed if dataset is large
    X_sample = X_test.iloc[:min(300, len(X_test))]

    # ── Initialise appropriate SHAP explainer ─────────────────────────────────
    print(f'[SHAP] Using model: {model_name}')
    if model_name in ('xgboost', 'lightgbm', 'random_forest'):
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
        # For tree models, shap_values may be a list [class0, class1]
        if isinstance(shap_values, list):
            shap_vals = shap_values[1]   # positive class (readmitted)
        else:
            shap_vals = shap_values
    else:
        # Logistic Regression — use LinearExplainer
        explainer = shap.LinearExplainer(model, X_sample)
        shap_vals = explainer.shap_values(X_sample)

    print(f'[SHAP] SHAP values computed: shape {shap_vals.shape}')

    # ── CHART 1: Beeswarm summary plot ────────────────────────────────────────
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_vals, X_sample,
        feature_names=feature_columns,
        plot_type='dot',
        max_display=15,
        show=False
    )
    plt.title('SHAP Beeswarm — Global Feature Impact on Readmission Risk', fontsize=12)
    plt.tight_layout()
    beeswarm_path = f'{output_dir}/shap_summary_beeswarm.png'
    plt.savefig(beeswarm_path, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close()
    print(f'  ✓ shap_summary_beeswarm.png')

    # ── CHART 2: Bar summary plot (mean |SHAP|) ────────────────────────────────
    plt.figure(figsize=(10, 7))
    shap.summary_plot(
        shap_vals, X_sample,
        feature_names=feature_columns,
        plot_type='bar',
        max_display=15,
        show=False
    )
    plt.title('SHAP Bar Chart — Mean |SHAP Value| per Feature', fontsize=12)
    plt.tight_layout()
    bar_path = f'{output_dir}/shap_summary_bar.png'
    plt.savefig(bar_path, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close()
    print(f'  ✓ shap_summary_bar.png')

    # ── CHART 3: Waterfall plots for 3 individual patients ────────────────────
    y_test      = state['y_test']
    y_pred      = model.predict(X_test)

    # Select high-risk patients: predicted readmitted
    high_risk_idx = np.where(y_pred == 1)[0][:3]
    if len(high_risk_idx) == 0:
        high_risk_idx = np.arange(min(3, len(X_sample)))

    waterfall_paths = []
    for rank, idx in enumerate(high_risk_idx):
        patient_shap = shap_vals[idx]
        actual       = int(y_test.iloc[idx]) if idx < len(y_test) else '?'
        predicted    = int(y_pred[idx])

        # Top 10 features for this patient
        top_idx = np.argsort(np.abs(patient_shap))[::-1][:10]
        top_feats = [feature_columns[i] for i in top_idx]
        top_vals  = patient_shap[top_idx]
        colors    = ['#E74C3C' if v > 0 else '#2ECC71' for v in top_vals]

        fig, ax = plt.subplots(figsize=(9, 5))
        bars = ax.barh(range(len(top_feats)), top_vals, color=colors, edgecolor='white')
        ax.set_yticks(range(len(top_feats)))
        ax.set_yticklabels(top_feats, fontsize=9)
        ax.axvline(x=0, color='black', linewidth=0.8)
        ax.set_xlabel('SHAP Value (impact on readmission probability)')
        ax.set_title(
            f'Patient {rank+1} — Local SHAP Explanation\n'
            f'Actual: {"Readmitted" if actual==1 else "Not Readmitted"} | '
            f'Predicted: {"HIGH RISK" if predicted==1 else "LOW RISK"}',
            fontsize=10
        )
        plt.tight_layout()
        wf_path = f'{output_dir}/shap_waterfall_patient_{rank+1}.png'
        plt.savefig(wf_path, bbox_inches='tight', dpi=150)
        plt.show()
        plt.close()
        waterfall_paths.append(wf_path)
        print(f'  ✓ shap_waterfall_patient_{rank+1}.png')

    # ── Print top global features ──────────────────────────────────────────────
    mean_shap = np.abs(shap_vals).mean(axis=0)
    top_global = sorted(
        zip(feature_columns, mean_shap),
        key=lambda x: x[1], reverse=True
    )[:10]
    print('\n[SHAP] Top 10 global features driving readmission risk:')
    for rank, (feat, val) in enumerate(top_global, 1):
        print(f'  {rank:>2}. {feat:<35} mean|SHAP| = {val:.4f}')

    print(f'\n[SHAP] ✓ Complete — outputs in {output_dir}/')

    return {
        **state,
        'shap_summary_path': beeswarm_path,
        'shap_values':       shap_vals,
    }

print('✓ shap_agent defined')

## Cell 11 — LIME Agent (Local Explanations)

In [ ]:
def lime_agent(state: AgentState) -> AgentState:
    """
    Generates LIME local explanations for 3 representative patients:
    - 1 high-risk correctly predicted readmission
    - 1 high-risk false negative (missed readmission)
    - 1 low-risk correctly predicted non-readmission

    Business insight: answers 'What specific factors made the model flag
    THIS patient as high-risk — and what should the care team act on?'
    """
    print('[LIME] Generating LIME local explanations...\n')

    model           = state['selected_model']
    X_train         = state['X_train']
    X_test          = state['X_test']
    y_test          = state['y_test']
    feature_columns = state['feature_columns']

    output_dir = 'lime_outputs'
    os.makedirs(output_dir, exist_ok=True)

    # ── Initialise LIME explainer ──────────────────────────────────────────────
    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data  = X_train.values,
        feature_names  = feature_columns,
        class_names    = ['Not Readmitted', 'Readmitted'],
        mode           = 'classification',
        random_state   = 42
    )

    y_pred = model.predict(X_test)
    y_test_arr = y_test.values

    # Select patients of interest
    tp_idx = np.where((y_pred == 1) & (y_test_arr == 1))[0]   # true positive
    fn_idx = np.where((y_pred == 0) & (y_test_arr == 1))[0]   # false negative
    tn_idx = np.where((y_pred == 0) & (y_test_arr == 0))[0]   # true negative

    patient_cases = [
        ('High-Risk (Correctly Flagged)',    tp_idx[0] if len(tp_idx) > 0 else 0),
        ('Missed Readmission (False Neg)',   fn_idx[0] if len(fn_idx) > 0 else 1),
        ('Low-Risk (Correctly Cleared)',     tn_idx[0] if len(tn_idx) > 0 else 2),
    ]

    lime_explanations = []

    for label, idx in patient_cases:
        print(f'[LIME] Explaining patient: {label} (test index {idx})')

        patient_data = X_test.iloc[idx].values
        actual       = int(y_test_arr[idx])
        predicted    = int(y_pred[idx])

        exp = explainer.explain_instance(
            data_row       = patient_data,
            predict_fn     = model.predict_proba,
            num_features   = 10,
            num_samples    = 1000
        )

        feature_weights = exp.as_list()
        risk_prob       = exp.predict_proba[1]

        # Plot
        feats = [fw[0] for fw in feature_weights]
        vals  = [fw[1] for fw in feature_weights]
        cols  = ['#E74C3C' if v > 0 else '#2ECC71' for v in vals]

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(range(len(feats)), vals, color=cols, edgecolor='white')
        ax.set_yticks(range(len(feats)))
        ax.set_yticklabels(feats, fontsize=8)
        ax.axvline(x=0, color='black', linewidth=0.8)
        ax.set_xlabel('LIME Weight (positive = increases readmission risk)')
        risk_label = 'HIGH RISK' if predicted == 1 else 'LOW RISK'
        ax.set_title(
            f'LIME — {label}\n'
            f'Actual: {"Readmitted" if actual==1 else "Not Readmitted"} | '
            f'Predicted: {risk_label} | Readmission Prob: {risk_prob:.1%}',
            fontsize=9
        )
        plt.tight_layout()
        safe_label = label.replace(' ', '_').replace('(', '').replace(')', '')
        lime_path = f'{output_dir}/lime_{safe_label}.png'
        plt.savefig(lime_path, bbox_inches='tight', dpi=150)
        plt.show()
        plt.close()

        lime_explanations.append({
            'label':            label,
            'test_idx':         int(idx),
            'actual':           actual,
            'predicted':        predicted,
            'readmission_prob': float(risk_prob),
            'feature_weights':  feature_weights,
            'plot_path':        lime_path,
        })
        print(f'  ✓ {lime_path} | Readmission prob: {risk_prob:.1%}')

    print(f'\n[LIME] ✓ Complete — 3 local explanations in {output_dir}/')

    return {
        **state,
        'lime_explanations': lime_explanations,
    }

print('✓ lime_agent defined')

## Cell 12 — PDP Agent (Partial Dependence Plots)

In [ ]:
def pdp_agent(state: AgentState) -> AgentState:
    """
    Generates Partial Dependence Plots for the top 4 most important features.

    Business insight: answers 'HOW does increasing/decreasing a feature
    affect readmission probability across the entire patient population?'
    """
    print('[PDP] Generating Partial Dependence Plots...\n')

    model           = state['selected_model']
    X_test          = state['X_test']
    feature_columns = state['feature_columns']
    shap_vals       = state.get('shap_values')

    output_dir = 'pdp_outputs'
    os.makedirs(output_dir, exist_ok=True)

    # ── Pick top 4 features by SHAP importance (or fallback to numeric) ────────
    if shap_vals is not None:
        mean_shap = np.abs(shap_vals).mean(axis=0)
        top_feature_indices = np.argsort(mean_shap)[::-1][:4]
        top_features = [feature_columns[i] for i in top_feature_indices]
    else:
        # Fallback: known high-impact clinical features
        preferred = [
            'num_medications', 'time_in_hospital',
            'number_inpatient', 'number_diagnoses'
        ]
        top_features = [f for f in preferred if f in feature_columns][:4]

    # Filter to features that exist in X_test
    top_features = [f for f in top_features if f in X_test.columns][:4]
    print(f'[PDP] Plotting PDPs for: {top_features}')

    pdp_paths = []

    for feat in top_features:
        feat_idx = list(X_test.columns).index(feat)

        try:
            pd_result = partial_dependence(
                estimator    = model,
                X            = X_test,
                features     = [feat_idx],
                kind         = 'average',
                percentiles  = (0.05, 0.95),
                grid_resolution = 50
            )

            grid_values = pd_result['grid_values'][0]
            avg_preds   = pd_result['average'][0]

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(grid_values, avg_preds, color='#E74C3C', linewidth=2.5)
            ax.fill_between(grid_values, avg_preds, alpha=0.15, color='#E74C3C')
            ax.set_xlabel(f'{feat}', fontsize=11)
            ax.set_ylabel('Partial Dependence\n(Readmission Probability)', fontsize=10)
            ax.set_title(
                f'Partial Dependence Plot — {feat}\n'
                f'How does {feat} affect readmission risk across all patients?',
                fontsize=10
            )
            ax.grid(axis='y', linestyle='--', alpha=0.4)
            plt.tight_layout()
            pdp_path = f'{output_dir}/pdp_{feat}.png'
            plt.savefig(pdp_path, bbox_inches='tight', dpi=150)
            plt.show()
            plt.close()
            pdp_paths.append(pdp_path)
            print(f'  ✓ pdp_{feat}.png')

        except Exception as e:
            print(f'  ⚠ Could not plot PDP for {feat}: {e}')

    print(f'\n[PDP] ✓ Complete — {len(pdp_paths)} plots in {output_dir}/')

    return {
        **state,
        'pdp_paths': pdp_paths,
    }

print('✓ pdp_agent defined')

## Cell 13 — Decision Agent (Clinical Risk Triage)

In [ ]:
def decision_agent(state: AgentState) -> AgentState:
    """
    Translates model predictions + LIME/SHAP insights into
    actionable clinical decisions for the care team.

    For each patient in the test set it assigns:
    - Risk tier: HIGH / MEDIUM / LOW
    - Intervention recommendation
    - Top 3 clinical drivers (from LIME explanations)

    Business value: bridges the gap between the ML model and
    the hospital discharge team. A nurse or case manager can
    act on this output without seeing any model internals.
    """
    print('[DECISION] Running clinical risk triage...\n')

    model            = state['selected_model']
    X_test           = state['X_test']
    y_test           = state['y_test']
    lime_explanations = state.get('lime_explanations', [])

    output_dir = 'decision_outputs'
    os.makedirs(output_dir, exist_ok=True)

    # ── Risk triage thresholds ────────────────────────────────────────────────
    # High: prob >= 0.5  →  flag for intensive discharge planning
    # Medium: 0.3 <= prob < 0.5 →  standard follow-up protocol
    # Low: prob < 0.3   →  routine discharge
    HIGH_THRESHOLD   = 0.50
    MEDIUM_THRESHOLD = 0.30

    def intervention(prob, tier):
        if tier == 'HIGH':
            return (
                'FLAG for intensive discharge planning. '
                'Schedule 48h post-discharge follow-up call. '
                'Review medication reconciliation. '
                'Assign dedicated case manager.'
            )
        elif tier == 'MEDIUM':
            return (
                'Schedule 7-day post-discharge GP visit. '
                'Send patient education materials. '
                'Confirm prescription filled on discharge.'
            )
        else:
            return 'Routine discharge. Standard follow-up protocol applies.'

    probs   = model.predict_proba(X_test)[:, 1]
    preds   = model.predict(X_test)
    actuals = y_test.values

    patient_risk_decisions = []

    for i in range(len(X_test)):
        p = float(probs[i])
        if p >= HIGH_THRESHOLD:
            tier = 'HIGH'
        elif p >= MEDIUM_THRESHOLD:
            tier = 'MEDIUM'
        else:
            tier = 'LOW'

        decision = {
            'patient_idx':   i,
            'readmission_probability': round(p, 4),
            'risk_tier':     tier,
            'predicted':     int(preds[i]),
            'actual':        int(actuals[i]),
            'intervention':  intervention(p, tier),
            'outcome_label': 'Readmitted' if actuals[i] == 1 else 'Not Readmitted',
        }
        patient_risk_decisions.append(decision)

    # ── Summary statistics ────────────────────────────────────────────────────
    tiers = [d['risk_tier'] for d in patient_risk_decisions]
    n_high   = tiers.count('HIGH')
    n_medium = tiers.count('MEDIUM')
    n_low    = tiers.count('LOW')
    total    = len(patient_risk_decisions)

    print('[DECISION] Risk tier distribution:')
    print(f'  🔴 HIGH   : {n_high:>5} patients ({n_high/total*100:.1f}%) — intensive intervention required')
    print(f'  🟡 MEDIUM : {n_medium:>5} patients ({n_medium/total*100:.1f}%) — standard follow-up')
    print(f'  🟢 LOW    : {n_low:>5} patients ({n_low/total*100:.1f}%) — routine discharge')

    # ── Sample output: top 5 high-risk patients ───────────────────────────────
    high_risk = sorted(
        [d for d in patient_risk_decisions if d['risk_tier'] == 'HIGH'],
        key=lambda x: x['readmission_probability'], reverse=True
    )[:5]

    print('\n[DECISION] Top 5 highest-risk patients:')
    print(f'  {"Patient":<10} {"Prob":<8} {"Tier":<8} {"Actual":<18} Intervention')
    print('  ' + '-' * 90)
    for d in high_risk:
        print(f'  #{d["patient_idx"]:<9} {d["readmission_probability"]:<8.1%} '
              f'{d["risk_tier"]:<8} {d["outcome_label"]:<18} '
              f'{d["intervention"][:55]}...')

    # ── Visualise risk distribution ───────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax1 = axes[0]
    tier_counts = {'HIGH': n_high, 'MEDIUM': n_medium, 'LOW': n_low}
    ax1.bar(tier_counts.keys(), tier_counts.values(),
            color=['#E74C3C', '#F39C12', '#2ECC71'], edgecolor='white')
    for k, v in tier_counts.items():
        ax1.text(k, v + 0.5, str(v), ha='center', fontweight='bold')
    ax1.set_title('Patient Risk Tier Distribution', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Number of Patients')

    ax2 = axes[1]
    ax2.hist(probs, bins=30, color='#3498DB', edgecolor='white', alpha=0.8)
    ax2.axvline(HIGH_THRESHOLD,   color='#E74C3C', linestyle='--', label=f'High threshold ({HIGH_THRESHOLD})')
    ax2.axvline(MEDIUM_THRESHOLD, color='#F39C12', linestyle='--', label=f'Medium threshold ({MEDIUM_THRESHOLD})')
    ax2.set_xlabel('Predicted Readmission Probability')
    ax2.set_ylabel('Number of Patients')
    ax2.set_title('Distribution of Readmission Probabilities', fontsize=11, fontweight='bold')
    ax2.legend(fontsize=8)

    plt.tight_layout()
    chart_path = f'{output_dir}/risk_tier_distribution.png'
    plt.savefig(chart_path, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close()

    # Save to CSV
    decisions_df = pd.DataFrame(patient_risk_decisions)
    csv_path = f'{output_dir}/patient_risk_decisions.csv'
    decisions_df.to_csv(csv_path, index=False)
    print(f'\n[DECISION] ✓ Decisions saved → {csv_path}')

    return {
        **state,
        'patient_risk_decisions': patient_risk_decisions,
    }

print('✓ decision_agent defined')

## Cell 14 — Business Agent (ROI & Downstream Scale)

In [ ]:
def business_agent(state: AgentState) -> AgentState:
    """
    Translates model performance + risk decisions into a C-suite business report.

    Quantifies:
    - Estimated cost savings from reduced readmissions
    - CMS penalty avoidance
    - ROI on deploying the model
    - Downstream scaling recommendations

    Business assumptions (US average benchmarks):
    - Average 30-day readmission cost:    $15,200 per event
    - CMS penalty per excess readmission: up to 3% of Medicare revenue
    - Model intervention success rate:    30% (conservative)
    - Annual hospital discharges modelled: scale from test set
    """
    print('[BUSINESS] Generating business impact report...\n')

    model                  = state['selected_model']
    selected_model_name    = state['selected_model_name']
    evaluation_results     = state['evaluation_results']
    patient_risk_decisions = state['patient_risk_decisions']
    feature_columns        = state['feature_columns']
    shap_vals              = state.get('shap_values')
    lime_explanations      = state.get('lime_explanations', [])

    # ── Financial constants ────────────────────────────────────────────────────
    READMISSION_COST_USD     = 15_200    # average cost per 30-day readmission
    INTERVENTION_SUCCESS     = 0.30     # fraction of high-risk patients averted
    MODEL_DEPLOY_COST_USD    = 50_000   # estimated annual model ops cost
    ANNUAL_SCALE_FACTOR      = 12       # test set → annual volume multiplier

    # ── Derive key numbers ─────────────────────────────────────────────────────
    metrics = evaluation_results[selected_model_name]
    recall  = metrics['recall']
    auc     = metrics.get('auc_roc', 0)
    f1      = metrics['f1_score']

    total_test = len(patient_risk_decisions)
    n_high_risk = sum(1 for d in patient_risk_decisions if d['risk_tier'] == 'HIGH')
    n_actual_readmits = sum(1 for d in patient_risk_decisions if d['actual'] == 1)
    n_caught = sum(
        1 for d in patient_risk_decisions
        if d['actual'] == 1 and d['predicted'] == 1
    )

    # Scale to annual
    annual_high_risk    = n_high_risk * ANNUAL_SCALE_FACTOR
    annual_readmits_caught = n_caught * ANNUAL_SCALE_FACTOR
    annual_prevented    = int(annual_readmits_caught * INTERVENTION_SUCCESS)
    gross_savings       = annual_prevented * READMISSION_COST_USD
    net_savings         = gross_savings - MODEL_DEPLOY_COST_USD
    roi_pct             = (net_savings / MODEL_DEPLOY_COST_USD) * 100

    # ── Top clinical drivers (from SHAP) ──────────────────────────────────────
    top_drivers = []
    if shap_vals is not None:
        mean_shap = np.abs(shap_vals).mean(axis=0)
        top_idx   = np.argsort(mean_shap)[::-1][:5]
        top_drivers = [(feature_columns[i], float(mean_shap[i])) for i in top_idx]

    # ── Build report ──────────────────────────────────────────────────────────
    report_lines = [
        '=' * 80,
        'HOSPITAL READMISSIONS — BUSINESS IMPACT REPORT',
        '=' * 80,
        '',
        '1. MODEL PERFORMANCE SUMMARY',
        '-' * 40,
        f'   Selected model : {selected_model_name}',
        f'   Recall         : {recall:.1%}  — fraction of actual readmissions the model catches',
        f'   AUC-ROC        : {auc:.4f}   — overall discriminative ability',
        f'   F1-Score       : {f1:.4f}   — balanced precision/recall',
        '',
        '2. PATIENT TRIAGE (TEST SET)',
        '-' * 40,
        f'   Total patients evaluated      : {total_test:>6,}',
        f'   Flagged HIGH risk             : {n_high_risk:>6,} ({n_high_risk/total_test*100:.1f}%)',
        f'   Actual readmissions in data   : {n_actual_readmits:>6,}',
        f'   Readmissions correctly caught : {n_caught:>6,}',
        '',
        '3. FINANCIAL IMPACT (ANNUAL PROJECTION)',
        '-' * 40,
        f'   Annual high-risk patients flagged      : ~{annual_high_risk:>6,}',
        f'   Annual readmissions model catches      : ~{annual_readmits_caught:>6,}',
        f'   Readmissions preventable (30% success) : ~{annual_prevented:>6,}',
        f'   Cost per readmission (US avg)          : ${READMISSION_COST_USD:>6,}',
        f'   Gross annual savings                   : ${gross_savings:>10,.0f}',
        f'   Model deployment & ops cost            : ${MODEL_DEPLOY_COST_USD:>10,.0f}',
        f'   NET annual savings                     : ${net_savings:>10,.0f}',
        f'   Estimated ROI                          : {roi_pct:>9.0f}%',
        '',
        '4. TOP CLINICAL DRIVERS (SHAP GLOBAL)',
        '-' * 40,
    ]

    clinical_guidance = {
        'num_medications':      'High medication count → pharmacy reconciliation at discharge',
        'time_in_hospital':     'Long stays → higher acuity; prioritise post-discharge support',
        'number_inpatient':     'Prior inpatient history → chronic condition management needed',
        'number_diagnoses':     'Multiple diagnoses → coordinate multi-specialty follow-up',
        'number_emergency':     'Emergency history → social/community care referral',
        'num_lab_procedures':   'High lab use → monitor closely; may indicate disease severity',
        'meds_per_day':         'High daily medication burden → adherence support program',
        'prior_utilisation':    'Heavy prior utilisation → case management referral',
    }

    for rank, (feat, val) in enumerate(top_drivers, 1):
        guidance = clinical_guidance.get(feat.split('_log')[0].split('_')[0] + '_' +
                                         '_'.join(feat.split('_')[1:]).split('_log')[0],
                                         clinical_guidance.get(feat, 'Review feature trend in discharge checklist'))
        report_lines.append(f'   {rank}. {feat:<35} mean|SHAP|={val:.4f}')
        report_lines.append(f'      → {guidance}')

    report_lines += [
        '',
        '5. DOWNSTREAM SCALING RECOMMENDATIONS',
        '-' * 40,
        '   A. IMMEDIATE (0–3 months)',
        '      • Integrate model output into the existing EHR discharge workflow.',
        '      • Display readmission risk score alongside vital signs on the ward dashboard.',
        '      • Pilot with a single ward; measure 30-day readmission rate vs control ward.',
        '',
        '   B. SHORT-TERM (3–12 months)',
        '      • Automate HIGH-risk patient flagging to case management team (email/SMS alert).',
        '      • Build SHAP explanation report into the discharge summary document.',
        '      • Retrain model quarterly on new patient data to avoid drift.',
        '',
        '   C. LONG-TERM (12+ months)',
        '      • Expand to all wards; integrate with community care platform.',
        '      • Add post-discharge wearable/telehealth data as new model features.',
        '      • Build A/B testing framework to measure intervention effectiveness by tier.',
        '      • Target: reduce 30-day readmission rate by 15–20% within 24 months.',
        '',
        '   D. REGULATORY & COMPLIANCE',
        '      • Document model decisions for CMS Hospital Readmissions Reduction Program audit.',
        '      • SHAP/LIME outputs serve as the explainability layer for clinical governance.',
        '      • Review model fairness across age, gender, and race subgroups annually.',
        '',
        '=' * 80,
        f'Report generated by Hospital Readmissions XAI Pipeline — {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}',
        '=' * 80,
    ]

    business_report = '\n'.join(report_lines)
    print(business_report)

    # ── Save report ───────────────────────────────────────────────────────────
    output_dir = 'business_outputs'
    os.makedirs(output_dir, exist_ok=True)
    report_path = f'{output_dir}/business_impact_report.txt'
    with open(report_path, 'w') as f:
        f.write(business_report)

    # ── ROI visualisation ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax1 = axes[0]
    labels   = ['Gross Savings', 'Deploy Cost', 'Net Savings']
    values   = [gross_savings, MODEL_DEPLOY_COST_USD, net_savings]
    bar_cols = ['#2ECC71', '#E74C3C', '#3498DB']
    bars = ax1.bar(labels, values, color=bar_cols, edgecolor='white')
    for bar, val in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                 f'${val:,.0f}', ha='center', fontsize=9, fontweight='bold')
    ax1.set_ylabel('USD (Annual)')
    ax1.set_title(f'Annual Financial Impact\nROI: {roi_pct:.0f}%', fontsize=11, fontweight='bold')
    ax1.grid(axis='y', linestyle='--', alpha=0.4)

    ax2 = axes[1]
    scale_months = [0, 3, 6, 12, 18, 24]
    readmit_rate = [100, 98, 94, 89, 85, 81]  # projected reduction (%)
    ax2.plot(scale_months, readmit_rate, 'o-', color='#3498DB', linewidth=2.5)
    ax2.fill_between(scale_months, readmit_rate, 80, alpha=0.1, color='#3498DB')
    ax2.set_xlabel('Months Post-Deployment')
    ax2.set_ylabel('Readmission Rate (% of baseline)')
    ax2.set_title('Projected Readmission Rate Reduction\n(Scaled Deployment Roadmap)',
                  fontsize=11, fontweight='bold')
    ax2.set_ylim(75, 105)
    ax2.grid(linestyle='--', alpha=0.4)
    for x, y in zip(scale_months, readmit_rate):
        ax2.annotate(f'{y}%', (x, y), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=8)

    plt.suptitle('Hospital Readmissions — Business Impact Dashboard',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    chart_path = f'{output_dir}/business_impact_chart.png'
    plt.savefig(chart_path, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close()

    print(f'\n[BUSINESS] ✓ Report saved → {report_path}')
    print(f'[BUSINESS] ✓ Chart saved  → {chart_path}')

    return {
        **state,
        'business_report': business_report,
    }

print('✓ business_agent defined')

## Cell 15 — Build Full LangGraph Workflow

In [ ]:
def build_workflow():
    """
    Compile the full 10-node LangGraph pipeline.

    Graph:
    START
      → eda_agent
      → clean_feature_engineering
      → train_models_agent
      → evaluate_models_agent
      → select_model_agent
      → shap_agent
      → lime_agent
      → pdp_agent
      → decision_agent
      → business_agent
    END
    """
    workflow = StateGraph(AgentState)

    # Register all nodes
    workflow.add_node('eda_agent',                  eda_agent)
    workflow.add_node('clean_feature_engineering',  clean_feature_engineering)
    workflow.add_node('train_models_agent',         train_models_agent)
    workflow.add_node('evaluate_models_agent',      evaluate_models_agent)
    workflow.add_node('select_model_agent',         select_model_agent)
    workflow.add_node('shap_agent',                 shap_agent)
    workflow.add_node('lime_agent',                 lime_agent)
    workflow.add_node('pdp_agent',                  pdp_agent)
    workflow.add_node('decision_agent',             decision_agent)
    workflow.add_node('business_agent',             business_agent)

    # Wire the edges
    workflow.add_edge(START,                        'eda_agent')
    workflow.add_edge('eda_agent',                  'clean_feature_engineering')
    workflow.add_edge('clean_feature_engineering',  'train_models_agent')
    workflow.add_edge('train_models_agent',         'evaluate_models_agent')
    workflow.add_edge('evaluate_models_agent',      'select_model_agent')
    workflow.add_edge('select_model_agent',         'shap_agent')
    workflow.add_edge('shap_agent',                 'lime_agent')
    workflow.add_edge('lime_agent',                 'pdp_agent')
    workflow.add_edge('pdp_agent',                  'decision_agent')
    workflow.add_edge('decision_agent',             'business_agent')
    workflow.add_edge('business_agent',             END)

    app = workflow.compile()
    return app

print('✓ build_workflow defined')

## Cell 16 — Run the Pipeline

In [ ]:
print('=' * 80)
print('HOSPITAL READMISSIONS — FULL XAI/MLI PIPELINE')
print('=' * 80)
print()

# ── Build initial state ────────────────────────────────────────────────────────
initial_state = {
    # upstream
    'raw_data':               df,
    'target_column':          'readmitted',
    'eda_charts_path':        None,
    # feature engineering
    'X_train':                None,
    'X_test':                 None,
    'y_train':                None,
    'y_test':                 None,
    'feature_columns':        None,
    # training
    'trained_models':         None,
    'training_summary':       None,
    'candidate_model_names':  None,
    # evaluation
    'evaluation_results':     None,
    'best_model_name':        None,
    # selection
    'selected_model_name':        None,
    'selected_model':             None,
    'selection_rationale':        None,
    'selection_scores':           None,
    'selection_comparison_path':  None,
    # XAI
    'shap_summary_path':      None,
    'shap_values':            None,
    'lime_explanations':      None,
    'pdp_paths':              None,
    # decision + business
    'patient_risk_decisions': None,
    'business_report':        None,
}

# ── Compile & invoke ───────────────────────────────────────────────────────────
print('[BUILD] Compiling 10-node LangGraph workflow...')
app = build_workflow()

print('[INVOKE] Running full pipeline...\n')
result = app.invoke(initial_state)

print()
print('=' * 80)
print('PIPELINE COMPLETE')
print('=' * 80)
print()
print(f'Selected model         : {result["selected_model_name"]}')
print(f'SHAP summary path      : {result["shap_summary_path"]}')
print(f'LIME explanations      : {len(result["lime_explanations"])} patients explained')
print(f'PDP plots              : {len(result["pdp_paths"])} features plotted')
print(f'Patients triaged       : {len(result["patient_risk_decisions"])}')
n_high = sum(1 for d in result['patient_risk_decisions'] if d['risk_tier'] == 'HIGH')
print(f'HIGH-risk patients     : {n_high}')
print()
print('Output folders:')
for folder in ['eda_outputs', 'evaluation_outputs', 'selection_outputs',
               'shap_outputs', 'lime_outputs', 'pdp_outputs',
               'decision_outputs', 'business_outputs']:
    if os.path.exists(folder):
        files = os.listdir(folder)
        print(f'  {folder}/  → {len(files)} file(s)')

## Cell 17 — Final Summary Print

In [ ]:
print('=' * 80)
print('FULL PIPELINE RESULTS SUMMARY')
print('=' * 80)

print('\n── MODEL RESULTS ──')
for model_name, metrics in result['evaluation_results'].items():
    marker = ' ← SELECTED' if model_name == result['selected_model_name'] else ''
    print(f'  {model_name:<28} | '
          f'recall={metrics["recall"]:.3f} | '
          f'f1={metrics["f1_score"]:.3f} | '
          f'auc={metrics.get("auc_roc", 0):.3f}{marker}')

print('\n── COMPOSITE SCORES ──')
for model_name, score in sorted(
    result['selection_scores'].items(), key=lambda x: x[1], reverse=True
):
    marker = ' ← SELECTED' if model_name == result['selected_model_name'] else ''
    print(f'  {model_name:<28} {score:.4f}{marker}')

print('\n── LIME EXPLANATIONS ──')
for exp in result['lime_explanations']:
    print(f'  [{exp["label"]}]')
    print(f'    Readmission prob : {exp["readmission_prob"]:.1%}')
    print(f'    Actual outcome   : {"Readmitted" if exp["actual"]==1 else "Not Readmitted"}')
    print(f'    Top feature      : {exp["feature_weights"][0][0]}')

print('\n── RISK TRIAGE ──')
decisions = result['patient_risk_decisions']
for tier, colour in [('HIGH', '🔴'), ('MEDIUM', '🟡'), ('LOW', '🟢')]:
    n = sum(1 for d in decisions if d['risk_tier'] == tier)
    pct = n / len(decisions) * 100
    print(f'  {colour} {tier:<8} {n:>5} patients ({pct:.1f}%)')